# Random Forest Hyperparameter Study

This notebook tunes a RandomForestClassifier with GridSearchCV using the selected and engineered features from the main modeling pipeline.

## Import Libraries

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split

## Configure Paths

In [ ]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from features.build_features import build_features

## Load Dataset

In [ ]:
data_path = project_root / "dataset" / "ml_dataset.csv"

if not data_path.exists():
    data_path = Path.cwd() / "ml_dataset.csv"

df = pd.read_csv(data_path)
df.head()

## Select And Engineer Features

In [ ]:
relevant_features = [
    "steps_per_hour",
    "calories_per_hour",
    "total_intensities_per_hour",
    "average_intensities_per_hour",
    "hour_of_day",
    "day_of_week",
    "steps_per_minute",
    "calories_per_minute",
    "total_steps_per_day",
    "total_calories_per_day",
    "days_recorded",
]

X = df[relevant_features].copy()
y = df.iloc[:, -1]

X = build_features(X)
X.head()

## Split Train And Test Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

## Run GridSearchCV

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"],
}

random_forest = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=random_forest,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

## Review Best Model

In [ ]:
print("Best parameters:")
print(grid_search.best_params_)
print(f"Best cross-validation F1 score: {grid_search.best_score_:.4f}")

best_rf_model = grid_search.best_estimator_

## Evaluate On Test Data

In [ ]:
test_predictions = best_rf_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print(f"Precision: {precision_score(y_test, test_predictions):.4f}")
print(f"Recall: {recall_score(y_test, test_predictions):.4f}")
print(f"F1 Score: {f1_score(y_test, test_predictions):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, test_predictions))

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_rf_model, X_test, y_test)
plt.title("Tuned Random Forest - Confusion Matrix")
plt.show()

## Inspect Grid Search Results

In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results = grid_results.sort_values("rank_test_score")

grid_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf",
        "param_class_weight",
    ]
].head(10)

## Plot Feature Importance

In [ ]:
feature_importances = pd.Series(
    best_rf_model.feature_importances_,
    index=X_train.columns,
).sort_values()

plt.figure(figsize=(10, 8))
feature_importances.plot(kind="barh")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.title("Tuned Random Forest Feature Importance")
plt.tight_layout()
plt.show()